In [ ]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import os
import pandas as pd
import numpy as np
import statistics
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import plotnine as p9

from src.clustering.utils import get_consensus_matrix

plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
eval_cols = ["LLM", "Clustering algorithm", "Clusters"]
comp_metrics = ["General performance score", "Stability", "Silhouette"]

In [ ]:
relevance_clusters_permutation = os.path.join("results_v2_permodality", "clustering", "relevance_clusters_permutation.csv")
relevance_clusters_permutation = pd.read_csv(relevance_clusters_permutation)
relevance_clusters_permutation = relevance_clusters_permutation.rename(columns={"Algorithm": "Clustering algorithm"})
relevance_clusters_permutation["LLM"] = relevance_clusters_permutation["LLM"].replace({
    "smb": "SMB-1.7B",
    "longformer": "Clinical-Longformer",
    "llama": "Llama3.3-70B",
    "nomic": "Nomic-Embed",
    "smb8b": "SMB-8B",
    "smb8b_qwen3_mo": "SMB-8B-Qwen3",
    "smb8b_llama3_cl": "SMB-8B-Llama3",
})
print(f"{relevance_clusters_permutation.shape=}")
relevance_clusters_permutation.head()

In [ ]:
embedding_clusters_permutation = os.path.join("results_v2_permodality", "clustering", "embedding_clusters_permutation.csv")
embedding_clusters_permutation = pd.read_csv(embedding_clusters_permutation)
embedding_clusters_permutation = embedding_clusters_permutation.rename(columns={"Algorithm": "Clustering algorithm"})
embedding_clusters_permutation["LLM"] = embedding_clusters_permutation["LLM"].replace({
    "smb": "SMB-1.7B",
    "longformer": "Clinical-Longformer",
    "llama": "Llama3.3-70B",
    "nomic": "Nomic-Embed",
    "smb8b": "SMB-8B",
    "smb8b_qwen3_mo": "SMB-8B-Qwen3",
    "smb8b_llama3_cl": "SMB-8B-Llama3",
})
embedding_clusters_permutation = embedding_clusters_permutation.merge(relevance_clusters_permutation, on=eval_cols + ["Fold"], validate="one_to_one")
reduced_embedding_clusters_permutation = embedding_clusters_permutation.groupby(eval_cols, as_index=False).mean(numeric_only=True)
print(f"{embedding_clusters_permutation.shape=}")
print(f"{reduced_embedding_clusters_permutation.shape=}")
embedding_clusters_permutation.head()

In [ ]:
cluster_stability = os.path.join("results_v2_permodality", "clustering", "cluster_stability.csv")
cluster_stability = pd.read_csv(cluster_stability)
cluster_stability = cluster_stability.rename(columns={"Algorithm": "Clustering algorithm"})
cluster_stability["LLM"] = cluster_stability["LLM"].replace({
    "smb": "SMB-1.7B",
    "longformer": "Clinical-Longformer",
    "llama": "Llama3.3-70B",
    "nomic": "Nomic-Embed",
    "smb8b": "SMB-8B",
    "smb8b_qwen3_mo": "SMB-8B-Qwen3",
    "smb8b_llama3_cl": "SMB-8B-Llama3",
})
cluster_stability["AMIs"] = cluster_stability["AMIs"].apply(eval)
cluster_stability = cluster_stability.rename(columns={"AMI": "Stability"})
print(f"{cluster_stability.shape=}")
cluster_stability.head()

In [ ]:
cluster_stability_wide = cluster_stability.explode('AMIs')
cluster_stability_wide = cluster_stability_wide.merge(reduced_embedding_clusters_permutation,
                                                      on=eval_cols, validate="many_to_many")
print(f"{cluster_stability_wide.shape=}")
cluster_stability_wide.head()

In [ ]:
cluster_stability = cluster_stability.merge(reduced_embedding_clusters_permutation, on=eval_cols, validate="one_to_one")
cluster_stability["General performance score"] = cluster_stability[["Stability", "Silhouette"]].clip(lower=0). \
    apply(statistics.harmonic_mean, axis=1)
cluster_stability_wide["General performance score"] = cluster_stability_wide[["AMIs", "Silhouette"]].clip(lower=0). \
    apply(statistics.harmonic_mean, axis=1)
print(f"{cluster_stability.shape=}")
cluster_stability.head()

In [ ]:
embedding_clusters = os.path.join("results_v2_permodality", "clustering", "embedding_clusters.csv")
embedding_clusters = pd.read_csv(embedding_clusters, index_col=0)
embedding_clusters.columns = pd.Series(embedding_clusters.columns).replace({
    "smb": "SMB-1.7B",
    "longformer": "Clinical-Longformer",
    "llama": "Llama3.3-70B",
    "nomic": "Nomic-Embed",
    "smb8b": "SMB-8B",
    "smb8b_qwen3_mo": "SMB-8B-Qwen3",
    "smb8b_llama3_cl": "SMB-8B-Llama3",
}, regex=True)
clustering_to_remove = []
for col_name, col_data in embedding_clusters.items():
    smallest_clusters = col_data.value_counts(normalize=True).iloc[-1]
    if smallest_clusters < 0.1:
        clustering_to_remove.append(col_name)
print(f"{len(clustering_to_remove)=}")

In [ ]:
cluster_stability = cluster_stability[
    cluster_stability["Clustering algorithm"].ne("SIMCADC") &
    cluster_stability["LLM"].ne("SMB-8B")
    ]
cluster_stability_wide = cluster_stability_wide[
    cluster_stability_wide["Clustering algorithm"].ne("SIMCADC") &
    cluster_stability_wide["LLM"].ne("SMB-8B")
    ]
embedding_clusters_permutation = embedding_clusters_permutation[
    embedding_clusters_permutation["Clustering algorithm"].ne("SIMCADC") &
    embedding_clusters_permutation["LLM"].ne("SMB-8B")
    ]

cluster_stability["Pipeline"] = cluster_stability["LLM"].astype(str) + "+" + cluster_stability["Clustering algorithm"].astype(str) + "+" + cluster_stability["Clusters"].astype(str)
cluster_stability = cluster_stability[~cluster_stability["Pipeline"].str.replace("+", "_").isin(clustering_to_remove)]

cluster_stability_wide["Pipeline"] = cluster_stability_wide["LLM"].astype(str) + "+" + cluster_stability_wide["Clustering algorithm"].astype(str) + "+" + cluster_stability_wide["Clusters"].astype(str)
cluster_stability_wide = cluster_stability_wide[~cluster_stability_wide["Pipeline"].str.replace("+", "_").isin(clustering_to_remove)]

embedding_clusters_permutation["Pipeline"] = embedding_clusters_permutation["LLM"].astype(str) + "+" + embedding_clusters_permutation["Clustering algorithm"].astype(str) + "+" + embedding_clusters_permutation["Clusters"].astype(str)
embedding_clusters_permutation = embedding_clusters_permutation[~embedding_clusters_permutation["Pipeline"].str.replace("+", "_").isin(clustering_to_remove)]

cluster_stability = cluster_stability[cluster_stability["Clusters"].le(7)]
cluster_stability_wide = cluster_stability_wide[cluster_stability_wide["Clusters"].le(7)]
embedding_clusters_permutation = embedding_clusters_permutation[embedding_clusters_permutation["Clusters"].le(7)]

print(f"{cluster_stability.shape=}")
print(f"{cluster_stability_wide.shape=}")
print(f"{embedding_clusters_permutation.shape=}")

cluster_stability.head()

In [ ]:
pval_comp_df = []
for y in comp_metrics:
    for x in eval_cols:
        custom_order = cluster_stability.groupby(x)[y].mean().sort_values(ascending=False).index
        name_1 = custom_order[0]
        name_2 = custom_order[1]
        perf1 = cluster_stability[cluster_stability[x].eq(name_1)][y]
        perf2 = cluster_stability[cluster_stability[x].eq(name_2)][y]
        p = stats.kruskal(perf1, perf2)[1]
        custom_order = cluster_stability.groupby(x)[comp_metrics[0]].mean().sort_values(ascending=False).index
        pval_dict = {
            "y": y,
            "x": x,
            "name_1": name_1,
            "name_2": name_2,
            "p": p,
            "x_label": (custom_order.get_loc(name_1) + custom_order.get_loc(name_2))/2+1,
        }
        pval_comp_df.append(pval_dict)
pval_comp_df = pd.DataFrame(pval_comp_df)
pval_comp_df["p"] = multipletests(pval_comp_df["p"], method='fdr_bh')[1]
pval_comp_df["y_start"] = 1.015
pval_comp_df["y_end"] = pval_comp_df["y_start"] - 0.02
pval_comp_df["y_label"] = pval_comp_df["y_start"] + 0.005
pval_comp_df["sig"] = pval_comp_df["p"].apply(lambda x: f"***" if x <= 0.001 else ("**" if x <= 0.01 else ("*" if x <= 0.05 else "")))         

In [ ]:
ps_overall = []
for y in comp_metrics:
    for x in eval_cols:
        if y == comp_metrics[0]:
            if x == "Clusters":
                custom_order = cluster_stability[x].sort_values(ascending=True).unique()
            else:
                custom_order = cluster_stability.groupby(x)[y].mean().sort_values(ascending=False).index.astype(str)
            cluster_stability[x] = pd.Categorical(cluster_stability[x], categories=custom_order, ordered=True)

        pval_comp_df_y = pval_comp_df[pval_comp_df["y"].eq(y) & pval_comp_df["x"].eq(x)]
        p = (
                p9.ggplot(cluster_stability.sort_values(by=x), p9.aes(x=x, y=y, fill=x))
                + p9.geom_violin(show_legend=False)
                + p9.geom_segment(
                        p9.aes(
                             x=p9.stage(x, after_stat="x-.45"),
                             xend=p9.stage(x, after_stat="x+.45"),
                             yend=p9.after_stat("y")
                        ),
                        stat="summary",
                        fun_y=np.mean,
                        size=1
                    )
                + p9.theme_classic()
                + p9.theme(axis_title_x=p9.element_blank(), plot_subtitle=p9.element_text(ha='center'))
                + p9.ylim((0,1.02))
                + p9.geom_segment(p9.aes(x='name_1', xend='name_2', y='y_start', yend='y_start'), data=pval_comp_df_y, color='black', inherit_aes=False)
                + p9.geom_segment(p9.aes(x='name_1', xend='name_1', y='y_start', yend='y_end'), data=pval_comp_df_y, color='black', inherit_aes=False)
                + p9.geom_segment(p9.aes(x='name_2', xend='name_2', y='y_start', yend='y_end'), data=pval_comp_df_y, color='black', inherit_aes=False)
                + p9.geom_text(p9.aes(x='x_label', y='y_label', label='sig'), data=pval_comp_df_y, inherit_aes=False)
        )
        if x == eval_cols[0]:
            p = p + p9.theme(axis_text_x=p9.element_text(angle=45))
        if x != eval_cols[0]:
            p = p + p9.theme(axis_title_y=p9.element_blank(), axis_text_y=p9.element_blank())
        if y == comp_metrics[0]:
            p = p + p9.labs(subtitle=x)
            if x == eval_cols[0]:
                p = p + p9.labs(tag="b")
            if x == eval_cols[1]:
                p = p + p9.labs(title="Aggregated performance per pipeline step")
        if y != comp_metrics[-1]:
            p = p + p9.theme(axis_text_x=p9.element_blank())
        ps_overall.append(p)

(ps_overall[0] | ps_overall[1] | ps_overall[2]) / \
(ps_overall[3] | ps_overall[4] | ps_overall[5]) / (ps_overall[6] | ps_overall[7] | ps_overall[8])

In [ ]:
pval_comp_df = []
x = "Pipeline"
for y in comp_metrics:
    if y == comp_metrics[0]:
        data_to_show = cluster_stability_wide
    elif y == comp_metrics[1]:
        data_to_show = cluster_stability_wide.drop(columns="Stability").rename(columns={"AMIs": comp_metrics[1]})
        data_to_show[y] = data_to_show[y].astype(float)
    elif y == comp_metrics[2]:
        data_to_show = embedding_clusters_permutation
    custom_order = data_to_show.groupby(x)[y].mean().sort_values(ascending=False).index
    name_1 = custom_order[0]
    name_2 = custom_order[1]
    perf1 = data_to_show[data_to_show[x].eq(name_1)][y]
    perf2 = data_to_show[data_to_show[x].eq(name_2)][y]
    p = stats.kruskal(perf1, perf2)[1]
    custom_order = cluster_stability.groupby(x)[comp_metrics[0]].mean().sort_values(ascending=False).index
    pval_dict = {
        "y": y,
        "name_1": name_1,
        "name_2": name_2,
        "p": p,
        "x_label": (custom_order.get_loc(name_1) + custom_order.get_loc(name_2))/2+1,
    }
    pval_comp_df.append(pval_dict)
pval_comp_df = pd.DataFrame(pval_comp_df)
pval_comp_df["p"] = multipletests(pval_comp_df["p"], method='fdr_bh')[1]
pval_comp_df["y_start"] = 1.015
pval_comp_df["y_end"] = pval_comp_df["y_start"] - 0.01
pval_comp_df["y_label"] = pval_comp_df["y_start"] + 0.005
pval_comp_df["sig"] = pval_comp_df["p"].apply(lambda x: f"***" if x <= 0.001 else ("**" if x <= 0.01 else ("*" if x <= 0.05 else "")))

In [ ]:
x = "Pipeline"
top_n = 10
ps_pipeline = []
for y in comp_metrics:
    title = None
    tag = None
    if y == comp_metrics[0]:
        data_to_show = cluster_stability_wide
        custom_order = data_to_show.groupby(x)[y].mean().sort_values(ascending=False).index.astype(str)
        data_to_show[x] = pd.Categorical(data_to_show[x], categories=custom_order, ordered=True)
        data_to_show = data_to_show[data_to_show[x].isin(custom_order[:top_n])].sort_values(by=x)
        plot_type = p9.geom_boxplot
        tag = "a"
    else:
        if y == comp_metrics[1]:
            data_to_show = cluster_stability_wide.drop(columns="Stability").rename(columns={"AMIs": comp_metrics[1]})
            data_to_show[y] = data_to_show[y].astype(float)
        elif y == comp_metrics[2]:
            data_to_show = embedding_clusters_permutation
        data_to_show[x] = pd.Categorical(data_to_show[x], categories=cluster_stability_wide[x].cat.categories, ordered=True)
        data_to_show = data_to_show[data_to_show[x].isin(cluster_stability_wide[x].cat.categories[:top_n])]
        plot_type = p9.geom_boxplot
    pval_comp_df_y = pval_comp_df[pval_comp_df["y"].eq(y)]
    p = (
            p9.ggplot(data_to_show, p9.aes(x=x, y=y))
            + plot_type(show_legend=False, color="#3b528b")
            + p9.theme_classic()
            + p9.theme(axis_text_x=p9.element_text(angle=75), axis_title_x=p9.element_blank(),
                       plot_title=p9.element_text(ha='center'))
            + p9.ylim((0.6,1.02))
            + p9.labs(tag=tag, title=title)
            + p9.geom_segment(p9.aes(x='name_1', xend='name_2', y='y_start', yend='y_start'), data=pval_comp_df_y, color='black', inherit_aes=False)
            + p9.geom_segment(p9.aes(x='name_1', xend='name_1', y='y_start', yend='y_end'), data=pval_comp_df_y, color='black', inherit_aes=False)
            + p9.geom_segment(p9.aes(x='name_2', xend='name_2', y='y_start', yend='y_end'), data=pval_comp_df_y, color='black', inherit_aes=False)
            + p9.geom_text(p9.aes(x='x_label', y='y_label', label='sig'), data=pval_comp_df_y, inherit_aes=False)
    )
    if y == comp_metrics[1]:
        p = p + p9.labs(title="Pipeline performance comparison")
    if y != comp_metrics[0]:
        p = p + p9.theme(axis_text_y=p9.element_blank())
    ps_pipeline.append(p)

ps_pipeline[0] | ps_pipeline[1] | ps_pipeline[2]

In [ ]:
composition = (ps_pipeline[0] | ps_pipeline[1] | ps_pipeline[2]) / \
(ps_overall[0] | ps_overall[1] | ps_overall[2]) / \
(ps_overall[3] | ps_overall[4] | ps_overall[5]) / (ps_overall[6] | ps_overall[7] | ps_overall[8]) + p9.theme(figure_size=(10, 13))

composition

In [ ]:
composition.save(os.path.join("figures", "Figure2.svg"))
composition.save(os.path.join("figures", "Figure2.pdf"))

In [ ]:
mask = (embedding_clusters_permutation["LLM"] == "SMB-8B-Qwen3") & (embedding_clusters_permutation["Clustering algorithm"] == "NEMO") & (embedding_clusters_permutation["Clusters"] == 2)
cl_result = embedding_clusters_permutation[mask]
for col in ["Labels", "IDs"]:
    cl_result[col] = cl_result[col].apply(lambda x: eval(x))

In [ ]:
consensus_matrix = get_consensus_matrix(
    cl_result,
    ids=list(set([x for patient_list in cl_result["IDs"].to_list() for x in patient_list])))
order_consensus_matrix = consensus_matrix.mean().sort_values().index
consensus_matrix = consensus_matrix.loc[order_consensus_matrix, order_consensus_matrix]
im = plt.imshow(consensus_matrix, cmap='hot', interpolation='nearest')
cbar = plt.colorbar(im)
plt.show()